# Предобработка данных

In [1]:
import pandas as pd
import numpy as np
import sklearn as sk

In [2]:
data = pd.read_csv('adult.csv')

In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 6.2 MB


## 1. Преобразования данных

Заменим значения колонки таргета '<=50k' и '>50k', на бинарные значения 0 и 1:

In [4]:
data['income'] = data['income'].map({'<=50K': 0, '>50K': 1})

Как мы узнали из EDA пропуски выглядят, как строка '?'. Заполним все пропуски в данных на строку 'Unknown':

In [5]:
data.replace('?', 'Unknown', inplace=True)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,Unknown,77053,HS-grad,9,Widowed,Unknown,Not-in-family,White,Female,0,4356,40,United-States,0
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,0
2,66,Unknown,186061,Some-college,10,Widowed,Unknown,Unmarried,Black,Female,0,4356,40,United-States,0
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,0
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,0
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,0
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,1
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,0


In [6]:
data[data == '?'].any().sum()

np.int64(0)

Пропусков в данных больше нет

Исходя из нашего EDA удалим ненужные признаки `education` и `fnlwgt`:

In [7]:
data.drop(['education', 'fnlwgt'], axis=1, inplace=True)

In [8]:
data.head(3)

,age,workclass,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,Unknown,9,Widowed,Unknown,Not-in-family,White,Female,0,4356,40,United-States,0
1,82,Private,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,0
2,66,Unknown,10,Widowed,Unknown,Unmarried,Black,Female,0,4356,40,United-States,0


Заменим признак `native.country` на новый признак сформированный на его основе - `is_usa`. Новый признак будет иметь только 3 возможных значения 'yes' - человек родом из USA, 'no' - человек родом из любой другой страны кроме USA и 'Unknown' - место рождения неизвестно.

In [9]:
data['is_usa'] = np.where(
    data['native.country'] == 'United-States', 'yes',
    np.where(data['native.country'] == 'Unknown', 'Unknown', 'no')
)
data.drop('native.country', axis=1, inplace=True)

## 2. Разделение данных

Разделим данные на train и test выборки, train - 80%, test - 20%

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(data.drop(columns=['income']), data['income'], test_size=0.2, random_state=52)
X_train_copy = X_train.copy()
X_test_copy = X_test.copy()

## 3. Кодирование категориальных признаков

Сохраним копии даных до кодирования для catboost, он сам хорошо кодирует их

In [12]:
X_train.to_csv('data/X_train_raw.csv', index=False)
X_test.to_csv('data/X_test_raw.csv', index=False)

Выделим все категориальные признаки

In [13]:
cat_cols = X_train.select_dtypes(include='str').columns.tolist()

In [14]:
print(f"Категориальные признаки: {', '.join(cat_cols)}")

Категориальные признаки: workclass, marital.status, occupation, relationship, race, sex, is_usa


Будем кодировать признаки с помощью one-hot encoding

In [15]:
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True, dtype=int)

In [16]:
X_train.head(3)

,age,education.num,capital.gain,capital.loss,hours.per.week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,...,relationship_Own-child,relationship_Unmarried,relationship_Wife,race_Asian-Pac-Islander,race_Black,race_Other,race_White,sex_Male,is_usa_no,is_usa_yes
21716,59,14,0,0,40,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,1
28602,23,9,0,0,40,0,0,1,0,0,...,1,0,0,0,0,0,1,1,0,1
2566,57,10,7688,0,45,0,0,1,0,0,...,0,0,0,0,0,0,1,1,0,1


Сохраним данные после кодирования признаков для дальнейшего использования там где не нужно масштабирование

In [17]:
X_train.to_csv('data/X_train_encoded.csv', index=False)
X_test.to_csv('data/X_test_encoded.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

## 4. Масштабирование числовых признаков

Некоторые модели (LogReg, SVM, KNN) чувствительны к масштабу признаков, поэтому перед обучением
приведём числовые признаки к единому масштабу через `StandardScaler`

In [18]:
from sklearn.preprocessing import StandardScaler

In [19]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled =  pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [20]:
X_train_scaled.head(3)

,age,education.num,capital.gain,capital.loss,hours.per.week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,...,relationship_Own-child,relationship_Unmarried,relationship_Wife,race_Asian-Pac-Islander,race_Black,race_Other,race_White,sex_Male,is_usa_no,is_usa_yes
0,1.498786,1.520923,-0.145313,-0.215487,-0.031025,-0.261482,-0.013856,-1.520489,-0.186659,-0.288699,...,-0.430770,-0.341955,-0.224987,-0.181306,-0.324388,-0.091868,0.412008,0.700541,-0.305685,0.33978
1,-1.138419,-0.422173,-0.145313,-0.215487,-0.031025,-0.261482,-0.013856,0.657683,-0.186659,-0.288699,...,2.321424,-0.341955,-0.224987,-0.181306,-0.324388,-0.091868,0.412008,0.700541,-0.305685,0.33978
2,1.352274,-0.033554,0.893630,-0.215487,0.374998,-0.261482,-0.013856,0.657683,-0.186659,-0.288699,...,-0.430770,-0.341955,-0.224987,-0.181306,-0.324388,-0.091868,0.412008,0.700541,-0.305685,0.33978


Сохраним отмасштабированные данные в файлах для дальнейшего использования

In [21]:
X_train_scaled.to_csv('data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('data/X_test_scaled.csv', index=False)

Всё, можно переходить к самим моделям